In [5]:
import cv2
import os
import numpy as np

input_folder = "anh_xlas"
output_folder = "output_cau2"

# Tạo thư mục đầu ra nếu chưa có
os.makedirs(output_folder, exist_ok=True)

# Lấy danh sách file ảnh hợp lệ
image_files = sorted([
    f for f in os.listdir(input_folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
])

print("Số ảnh tìm thấy:", len(image_files))

Số ảnh tìm thấy: 10


In [6]:
def manual_filter2d_box(src_img, kernel_size):
    h, w = src_img.shape
    pad = kernel_size // 2

    # Tạo ảnh đệm mở rộng (padding = 0)
    padded = np.zeros((h + 2 * pad, w + 2 * pad), dtype=np.float32)
    for r in range(h):
        for c in range(w):
            padded[r + pad, c + pad] = src_img[r, c]

    out = np.zeros((h, w), dtype=np.uint8)
    k_total = kernel_size * kernel_size

    # Trượt cửa sổ tính trung bình cộng
    for r in range(h):
        for c in range(w):
            pixel_sum = 0.0
            for kr in range(kernel_size):
                for kc in range(kernel_size):
                    pixel_sum += padded[r + kr, c + kc]
            out[r, c] = int(pixel_sum / k_total)

    return out


def manual_median_blur(src_img, kernel_size):
    h, w = src_img.shape
    pad = kernel_size // 2

    # Tạo ảnh đệm mở rộng (padding = 0)
    padded = np.zeros((h + 2 * pad, w + 2 * pad), dtype=np.uint8)
    for r in range(h):
        for c in range(w):
            padded[r + pad, c + pad] = src_img[r, c]

    out = np.zeros((h, w), dtype=np.uint8)
    k_total = kernel_size * kernel_size

    # Trượt cửa sổ tìm số chính giữa
    for r in range(h):
        for c in range(w):
            window_pixels = []
            for kr in range(kernel_size):
                for kc in range(kernel_size):
                    window_pixels.append(padded[r + kr, c + kc])

            # Sắp xếp nổi bọt thủ công (Bubble Sort)
            for i in range(k_total):
                for j in range(0, k_total - i - 1):
                    if window_pixels[j] > window_pixels[j + 1]:
                        window_pixels[j], window_pixels[j + 1] = window_pixels[j + 1], window_pixels[j]

            # Lấy giá trị trung vị ở giữa mảng
            out[r, c] = window_pixels[k_total // 2]

    return out

In [ ]:
for filename in image_files:
    image_path = os.path.join(input_folder, filename)
    img = cv2.imread(image_path)
    if img is None:
        print(f"Không đọc được {filename}")
        continue

    # 1. Chuyển ảnh xám (Sử dụng hàm hệ thống)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2.Tính I1, I2 và vùng đệm temp (bộ lọc trung bình)
    I1 = manual_filter2d_box(gray, 3)
    I2 = manual_filter2d_box(gray, 5)
    temp = manual_filter2d_box(gray, 7)

    # 3. Giảm mẫu với Stride = 2 tạo I3
    h3, w3 = (temp.shape[0] + 1) // 2, (temp.shape[1] + 1) // 2
    I3 = np.zeros((h3, w3), dtype=np.uint8)
    for r in range(h3):
        for c in range(w3):
            I3[r, c] = temp[r * 2, c * 2]

    # 4. Bộ lọc trung vị tạo I4 và I5
    I4 = manual_median_blur(I3, 3)
    I5 = manual_median_blur(I1, 5)

    # 5. Đồng bộ kích thước (Padding góc) cho pad_I4 và pad_I5
    h4, w4 = I4.shape
    h5, w5 = I5.shape
    new_h = h4 if h4 > h5 else h5
    new_w = w4 if w4 > w5 else w5

    pad_I4 = np.zeros((new_h, new_w), dtype=np.uint8)
    pad_I5 = np.zeros((new_h, new_w), dtype=np.uint8)

    for r in range(h4):
        for c in range(w4):
            pad_I4[r, c] = I4[r, c]

    for r in range(h5):
        for c in range(w5):
            pad_I5[r, c] = I5[r, c]

    # 6. So sánh giá trị từng pixel tạo I6 (thay thế np.where)
    I6 = np.zeros((new_h, new_w), dtype=np.uint8)
    for r in range(new_h):
        for c in range(new_w):
            if pad_I4[r, c] > pad_I5[r, c]:
                I6[r, c] = 0
            else:
                I6[r, c] = pad_I5[r, c]

    # 7. Lưu tất cả kết quả xuống thư mục đầu ra
    name = os.path.splitext(filename)[0]
    cv2.imwrite(os.path.join(output_folder, f"Gray_{name}.jpg"), gray)
    cv2.imwrite(os.path.join(output_folder, f"I1_{name}.jpg"), I1)
    cv2.imwrite(os.path.join(output_folder, f"I2_{name}.jpg"), I2)
    cv2.imwrite(os.path.join(output_folder, f"I3_{name}.jpg"), I3)
    cv2.imwrite(os.path.join(output_folder, f"I4_{name}.jpg"), pad_I4)
    cv2.imwrite(os.path.join(output_folder, f"I5_{name}.jpg"), pad_I5)
    cv2.imwrite(os.path.join(output_folder, f"I6_{name}.jpg"), I6)

    print(f"Processing {filename} ... Done!")

print(f"\nFinished processing {len(image_files)} images.")

Processing image_01.jpg ... Done!
Processing image_02.jpg ... Done!
Processing image_03.jpg ... Done!
Processing image_04.jpg ... Done!
Processing image_05.jpg ... Done!
Processing image_06.jpg ... Done!
Processing image_07.jpg ... Done!
